In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

# 1. Define the parameters for a single generator
class Generator(BaseModel):
    name: str
    cost_0: float
    cost_1: float
    cost_2: float
    max_capacity_mw: float
    min_capacity_mw: float = 0.0

# 2. Define the overall problem structure
class DispatchProblem(BaseModel):
    objective_type: str  # e.g., "minimize cost"
    total_demand_mw: float
    generators: list[Generator]

print("here")

here


In [8]:
# 1. Load the hidden API key from your .env file
load_dotenv()

# 2. Initialize the Gemini client 
client = genai.Client()

# 3. Define the plain-text problem (Unstructured Data)
prompt_text = """
The grid operator needs to meet a total demand of 300 MW. There are three generators available, each with a quadratic cost curve.

Generator 1 is a large baseload plant. Its quadratic coefficient 'a' is 0.05, its linear coefficient 'b' is 10, and its no-load cost 'c' is 500. It has a maximum capacity of 250 MW and a minimum capacity of 50 MW.
Generator 2 is a mid-merit gas plant. Its '2' coefficient is 0.05, '1' is 20, and '0' is 200. It has a maximum capacity of 200 MW and a minimum capacity of 50 MW.
Generator 3 is a fast-ramping peaking unit. Its cost function is 0.1*P^2 + 28*P + 50. It has a maximum capacity of 100 MW. It can be turned off completely, so it has no minimum capacity constraint.

Minimize the total system cost.
"""

# 4. Ask Gemini to read the text and fill out the Pydantic form
print("Sending text to Gemini...")
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=DispatchProblem,
        temperature=0.0,
    ),
)

# 5. Convert the JSON response back into our Pydantic object
extracted_data = DispatchProblem.model_validate_json(response.text)

print("\n--- Extraction Success! ---")
print(f"Objective: {extracted_data.objective_type}")
print(f"Demand: {extracted_data.total_demand_mw} MW")
for gen in extracted_data.generators:
    print(f"Found {gen.name}: Min {gen.min_capacity_mw}MW, Max {gen.max_capacity_mw}MW, Cost {gen.cost_2}*P^2 + {gen.cost_1}*P + {gen.cost_0}")

Sending text to Gemini...

--- Extraction Success! ---
Objective: minimize_cost
Demand: 300.0 MW
Found Generator 1: Min 50.0MW, Max 250.0MW, Cost 0.05*P^2 + 10.0*P + 500.0
Found Generator 2: Min 50.0MW, Max 200.0MW, Cost 0.05*P^2 + 20.0*P + 200.0
Found Generator 3: Min 0.0MW, Max 100.0MW, Cost 0.1*P^2 + 28.0*P + 50.0


In [9]:
import pyomo.environ as pyo
model = pyo.ConcreteModel()
gen_names = [g.name for g in extracted_data.generators]
print(gen_names)
model.generator = pyo.Set(initialize = gen_names)
model.total_demand_mw = pyo.Param(initialize = extracted_data.total_demand_mw)
model.gen_power = pyo.Var(model.generator)

costs = {g.name: [g.cost_0, g.cost_1, g.cost_2] for g in extracted_data.generators}
model.cost = pyo.Param(model.generator, initialize=costs)


min_capacity = {g.name: g.min_capacity_mw for g in extracted_data.generators}
model.min_capicity = pyo.Param(model.generator, initialize=min_capacity)
max_capacity = {g.name: g.max_capacity_mw for g in extracted_data.generators}
model.max_capicity = pyo.Param(model.generator, initialize=max_capacity)

def max_rule(model, g):
    return model.gen_power[g] <= model.max_capicity[g]
def min_rule(model, g):
    return model.gen_power[g] >= model.min_capicity[g]
model.c1 = pyo.Constraint(expr = sum(model.gen_power[g] for g in model.generator)== model.total_demand_mw)
model.c2 = pyo.Constraint(model.generator, rule = max_rule)
model.c3 = pyo.Constraint(model.generator, rule = min_rule)

model.f1 = pyo.Objective(expr = sum(model.cost[g][2] * model.gen_power[g]**2 + model.cost[g][1] * model.gen_power[g] + model.cost[g][0] for g in model.generator), sense=pyo.minimize)

solver = pyo.SolverFactory('ipopt')
results = solver.solve(model)
for g in model.generator:
    print(f"{g}: {pyo.value(model.gen_power[g])} MW")



['Generator 1', 'Generator 2', 'Generator 3']


Generator 1: 195.99999999913095 MW
Generator 2: 95.99999999972721 MW
Generator 3: 8.000000001141839 MW
